# Riemann data pipeline — consolidated 00–05

Single-run Colab pipeline. The notebook is intentionally restart-free: `pip install -e .` is followed by a normal import in the same runtime.

Stages: 00 environment → 01 acquire → 02 describe → 03 unfold → 04 surrogates → 05 compare.

**Artifact rule:** existing artifacts are never overwritten. Existing raw/derived artifacts are verified against `data/manifest.json`. A missing derived artifact is created once, then hashed and added to the local manifest; any subsequent mismatch is a hard error.

## 00 — Environment


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

REPO_BASE = Path("/content/nicht-riemann-data")
REPO_URL = "https://github.com/nicht-organization/nicht-riemann-data.git"

if not REPO_BASE.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_BASE)], check=True)
elif not (REPO_BASE / ".git").is_dir():
    raise RuntimeError(f"Path exists but is not a Git repository: {REPO_BASE}")

os.chdir(REPO_BASE)
print("cwd:", Path.cwd())
print(sys.version)

subprocess.run(["git", "status", "--short"], check=True)
subprocess.run(["git", "branch", "--show-current"], check=True)

# Install into this interpreter; no runtime exit/restart is required.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

from nicht_riemann_data.transforms import spacings, normalized_spacings
from nicht_riemann_data.diagnostics import describe

print("Package import: OK")


## Integrity helpers


In [ ]:
MANIFEST_FILE = Path("data/manifest.json")
DATA_DIR = Path("data/raw")
DERIVED_DIR = Path("data/derived")

assert MANIFEST_FILE.is_file(), f"Missing manifest: {MANIFEST_FILE}"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def load_manifest() -> dict:
    with MANIFEST_FILE.open("r", encoding="utf-8") as f:
        return json.load(f)

def save_manifest(manifest: dict) -> None:
    tmp = MANIFEST_FILE.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
        f.write("\n")
    tmp.replace(MANIFEST_FILE)

manifest = load_manifest()
print("manifest:", MANIFEST_FILE)


## 01 — Acquire

The raw dataset is required locally. It is never downloaded over an existing file.


In [ ]:
import numpy as np

DATASET = "zeros1"
RAW_FILE = DATA_DIR / DATASET
raw_spec = manifest["datasets"]["odlyzko_zeros1"]

assert RAW_FILE.is_file(), f"Missing raw artifact: {RAW_FILE}"

actual_bytes = RAW_FILE.stat().st_size
actual_sha256 = sha256(RAW_FILE)

assert actual_bytes == raw_spec["raw_bytes"], (
    f"Raw artifact size mismatch: {actual_bytes} != {raw_spec['raw_bytes']}"
)
assert actual_sha256 == raw_spec["sha256"], (
    f"Raw artifact SHA-256 mismatch: {actual_sha256} != {raw_spec['sha256']}"
)

print(f"Verified raw artifact: {RAW_FILE}")
print("bytes:", actual_bytes)
print("SHA-256:", actual_sha256)


In [ ]:
gamma = np.loadtxt(RAW_FILE, dtype=np.float64)

assert gamma.ndim == 1
assert gamma.dtype == np.float64
assert np.all(np.isfinite(gamma))
assert np.all(np.diff(gamma) > 0)

delta = spacings(gamma)
assert delta.shape == (len(gamma) - 1,)
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)

print("zeros    :", len(gamma))
print("spacings :", len(delta))
print("first    :", delta[:10])
print("gamma:", describe(gamma))
print("delta:", describe(delta))


## 02 — Describe


In [ ]:
gamma_range = gamma[-1] - gamma[0]
mean_spacing = float(np.mean(delta))
range_per_spacing = gamma_range / len(delta)

print("range:", gamma_range)
print("mean spacing:", mean_spacing)
print("range / number of spacings:", range_per_spacing)
assert np.isclose(mean_spacing, range_per_spacing)

percentile_levels = [0, 1, 5, 25, 50, 75, 95, 99, 100]
for p, value in zip(percentile_levels, np.percentile(delta, percentile_levels)):
    print(f"{p:>3}% : {value:.12f}")

BLOCK_SIZE = 1000
num_blocks = len(delta) // BLOCK_SIZE
block_means = np.array([
    np.mean(delta[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE])
    for i in range(num_blocks)
])
remainder_delta = delta[num_blocks * BLOCK_SIZE:]
print("block size:", BLOCK_SIZE)
print("full blocks:", num_blocks)
print("remainder:", len(remainder_delta))
if len(block_means):
    print("local mean min/max/std:", block_means.min(), block_means.max(), block_means.std())

predicted_global = gamma[0] + np.arange(len(gamma)) * mean_spacing
residual_global = gamma - predicted_global
print("global baseline residual std:", np.std(residual_global))


## 03 — Unfold

If `data/derived/unfolded_spacings.float64` already exists, it is **never overwritten**: size, SHA-256, dtype, length, finiteness, and positivity are checked against the manifest.

If it is absent, the notebook creates it from the verified raw spacings using the package's `normalized_spacings`, verifies the resulting bytes against the manifest if an entry already exists, and otherwise adds the new artifact entry to the **local** manifest.


In [ ]:
UNFOLDED_FILE = DERIVED_DIR / "unfolded_spacings.float64"
derived = manifest.setdefault("derived", {})
unfolded_spec = derived.get("unfolded_spacings")

if UNFOLDED_FILE.exists():
    assert unfolded_spec is not None, (
        f"Existing artifact has no manifest entry: {UNFOLDED_FILE}"
    )
else:
    unfolded_new = normalized_spacings(gamma)
    assert unfolded_new.shape == delta.shape
    assert unfolded_new.dtype == np.float64
    assert np.all(np.isfinite(unfolded_new))
    assert np.all(unfolded_new > 0)

    unfolded_new.tofile(UNFOLDED_FILE)

    created_bytes = UNFOLDED_FILE.stat().st_size
    created_sha256 = sha256(UNFOLDED_FILE)
    if unfolded_spec is not None:
        assert created_bytes == unfolded_spec["bytes"], (
            f"Created unfolded artifact size mismatch: {created_bytes} != {unfolded_spec['bytes']}"
        )
        assert created_sha256 == unfolded_spec["sha256"], (
            f"Created unfolded artifact SHA-256 mismatch: {created_sha256} != {unfolded_spec['sha256']}"
        )
    else:
        unfolded_spec = {
            "source_notebook": "notebooks/riemann_pipeline.ipynb",
            "local_file": str(UNFOLDED_FILE),
            "records": int(unfolded_new.size),
            "representation": "binary IEEE-754 float64",
            "quantity": "unfolded nearest-neighbor spacings",
            "dtype": "float64",
            "bytes": int(created_bytes),
            "sha256": created_sha256,
        }
        derived["unfolded_spacings"] = unfolded_spec
        save_manifest(manifest)
        print("Created manifest entry for new derived artifact.")

expected_bytes = unfolded_spec["bytes"]
expected_sha256 = unfolded_spec["sha256"]
actual_bytes = UNFOLDED_FILE.stat().st_size
actual_sha256 = sha256(UNFOLDED_FILE)

assert actual_bytes == expected_bytes, (
    f"Existing unfolded artifact size mismatch: {actual_bytes} != {expected_bytes}"
)
assert actual_sha256 == expected_sha256, (
    f"Existing unfolded artifact SHA-256 mismatch: {actual_sha256} != {expected_sha256}"
)

unfolded = np.fromfile(UNFOLDED_FILE, dtype=np.float64)
assert unfolded.ndim == 1
assert unfolded.size == len(delta)
assert unfolded.dtype == np.float64
assert np.all(np.isfinite(unfolded))
assert np.all(unfolded > 0)

print(f"Verified unfolded artifact: {UNFOLDED_FILE}")
print("bytes:", actual_bytes)
print("SHA-256:", actual_sha256)
print("records:", unfolded.size)
print("mean:", np.mean(unfolded))
print("std :", np.std(unfolded))


## 04 — Surrogates

No reacquisition or reunfolding. Surrogates are generated directly from the verified 03 artifact.

In [ ]:
SEED = 20260831
rng = np.random.default_rng(SEED)

surrogate_shuffle = unfolded.copy()
rng.shuffle(surrogate_shuffle)
assert np.array_equal(np.sort(surrogate_shuffle), np.sort(unfolded))

surrogate_iid = rng.choice(unfolded, size=len(unfolded), replace=True)
assert surrogate_iid.shape == unfolded.shape
assert np.all(np.isfinite(surrogate_iid))
assert np.all(surrogate_iid > 0)

surrogate_uniform = rng.uniform(0.0, 2.0, size=len(unfolded))
assert surrogate_uniform.shape == unfolded.shape
assert np.all(np.isfinite(surrogate_uniform))
assert np.all(surrogate_uniform >= 0)

datasets = {
    "observed": unfolded,
    "shuffle": surrogate_shuffle,
    "iid": surrogate_iid,
    "uniform": surrogate_uniform,
}

for name, values in datasets.items():
    assert values.ndim == 1
    assert len(values) == len(unfolded)
    assert np.all(np.isfinite(values))
    assert np.all(values >= 0)

print("datasets:", ", ".join(datasets))
print("N:", len(unfolded))


## 05 — Compare


In [ ]:
print("=== distribution summary ===")
for name, values in datasets.items():
    print(
        f"{name:>8}: mean={np.mean(values):.8f}  "
        f"std={np.std(values):.8f}  "
        f"min={np.min(values):.8f}  "
        f"max={np.max(values):.8f}"
    )


In [ ]:
PERCENTILES = [1, 5, 25, 50, 75, 95, 99]
print("=== quantiles ===")
for name, values in datasets.items():
    q = np.percentile(values, PERCENTILES)
    print(name, {p: float(v) for p, v in zip(PERCENTILES, q)})


In [ ]:
print("=== observed vs shuffled invariant ===")
assert np.isclose(np.mean(datasets["observed"]), np.mean(datasets["shuffle"]))
assert np.isclose(np.std(datasets["observed"]), np.std(datasets["shuffle"]))
print("shuffle preserves the observed sample mean and standard deviation.")


## Pipeline complete

All stages completed without a forced runtime exit. Any artifact integrity failure raises immediately and stops execution.
